# RHNA & Housing Production

RHNA targets and housing production (permits/completions) for the 18
incorporated jurisdictions in San Diego County plus the County itself,
pulled from HCD, DOF, City of San Diego, and Census sources.

Target year: 2025 for RHNA/APR/DOF/permits. ACS uses the 2020-2024
5-year vintage (its most recent release).


## Setup

In [ ]:
%pip install pandas numpy requests openpyxl

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "van's work"
        if (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError(
        "Could not locate workstream root (no notebooks/ folder found nearby)."
    )


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Workstream root:", ROOT)


In [ ]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "San Diego County"


In [ ]:
# Variants of the county's name that all appear across HCD/DOF sources --
# "San Diego County" as a jurisdiction (in APR, RHNA) represents only the
# unincorporated portion of the county, not the whole region including
# cities. Renamed to "Unincorporated San Diego County" to make that
# explicit rather than leaving it ambiguous as just "San Diego County".
COUNTY_NAME_VARIANTS = {
    "san diego county", "county of san diego", "s d county",
    "county san diego", "unincorporated", "unincorporated san diego county",
}


def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    if s in COUNTY_NAME_VARIANTS:
        return "unincorporated san diego county"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


## APR (permits, entitlements, completions)

In [ ]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


In [ ]:
print(apr_raw.columns.tolist())


**Development stages tracked here, kept strictly separate:**

| Stage | Source | Metric |
|---|---|---|
| Application submitted | APR Table A | `application_units_total` |
| Entitlement | APR Table A2 (unprefixed `*_INCOME_*` columns; cross-checked against HCD's own auto-populated `NO_ENTITLEMENTS` total) | `ent_units_total` |
| Building permit | APR Table A2 (`BP_*_INCOME` columns; cross-checked against `NO_BUILDING_PERMITS`) | `bp_units_total` |
| Completion (certificate of occupancy) | APR Table A2 (`CO_*_INCOME` columns; cross-checked against `NO_OTHER_FORMS_OF_READINESS`) | `co_units_total` |

**Units under construction** is the one stage from the dashboard reorg
guidance that is **not tracked anywhere in HCD's APR data** -- no table
captures this milestone.

These four stages are never summed into one another or into a single
"production" figure -- a project can appear as an application one year,
entitled the next, permitted the year after that, so combining them would
double-count the same physical units across stages.

In [ ]:
JURISDICTION_COL = "JURIS_NAME"
YEAR_COL = "YEAR"

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


In [ ]:
# Table A2 tracks THREE separate stages, not two: unprefixed *_INCOME_*
# columns are entitlement-stage units (paired with ENT_APPROVE_DT1), BP_*
# are building permits, CO_* are completions. Kept as three fully separate
# metrics per the reorg requirement -- never summed into one another.
ENT_INCOME_COLS = [
    c for c in sd_apr_target_year.columns
    if "INCOME" in c.upper() and not c.upper().startswith("BP_") and not c.upper().startswith("CO_")
]
BP_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("BP_") and "INCOME" in c.upper()]
CO_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("CO_") and "INCOME" in c.upper()]

sd_apr_target_year["ent_units_row"] = sd_apr_target_year[ENT_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["bp_units_row"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_row"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_ent = [c for c in ENT_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["ent_affordable_row"] = (
    sd_apr_target_year["ent_units_row"] - sd_apr_target_year[above_mod_ent].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["bp_affordable_row"] = (
    sd_apr_target_year["bp_units_row"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_row"] = (
    sd_apr_target_year["co_units_row"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)

# The raw file is row-level (one row per project/address) -- group up to
# jurisdiction-year before this becomes a usable production metric.
production_by_year = (
    sd_apr_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        year=(YEAR_COL, "first"),
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
    )
)
production_by_year["ent_affordable_share"] = production_by_year["ent_affordable_total"] / production_by_year["ent_units_total"]
production_by_year["bp_affordable_share"] = production_by_year["bp_affordable_total"] / production_by_year["bp_units_total"]
production_by_year["co_affordable_share"] = production_by_year["co_affordable_total"] / production_by_year["co_units_total"]
production_by_year


### Cross-check against HCD's own auto-populated totals

`NO_ENTITLEMENTS`, `NO_BUILDING_PERMITS`, and `NO_OTHER_FORMS_OF_READINESS`
are not flags despite the naming -- per HCD's own APR instructions, these
are auto-populated fields meaning "**N**umber **O**f Entitlements/Permits/etc.",
calculated by HCD from the same per-tier income columns this notebook sums
independently. If our totals match HCD's own auto-populated totals, that's
strong confirmation the entitlement/permit/completion calculations are
using the correct unit-count fields, not an assumption based on dates or
flag presence.

In [ ]:
validation_cols = {
    "ent_units_total": "NO_ENTITLEMENTS",
    "bp_units_total": "NO_BUILDING_PERMITS",
    "co_units_total": "NO_OTHER_FORMS_OF_READINESS",
}

hcd_totals = sd_apr_target_year.groupby("jur_clean", as_index=False)[list(validation_cols.values())].sum()
stage_check = production_by_year.merge(hcd_totals, on="jur_clean")

for our_col, hcd_col in validation_cols.items():
    stage_check[f"{our_col}_diff"] = stage_check[our_col] - stage_check[hcd_col]

diff_cols = [f"{c}_diff" for c in validation_cols]
print("Max absolute difference per stage:")
print(stage_check[diff_cols].abs().max())
stage_check[["jur_clean"] + list(validation_cols.keys()) + list(validation_cols.values()) + diff_cols]


### Historical range check (2018-2025)

APR data collection began in 2018 -- pull the full history rather than
just the target year, and confirm every year is actually present for San
Diego County before treating this as a complete trend, rather than
assuming it is.

In [ ]:
APR_START_YEAR = 2018   # APR data collection began in 2018
APR_YEARS = list(range(APR_START_YEAR, TARGET_YEAR + 1))

ENT_INCOME_COLS_ALL = [
    c for c in sd_apr.columns
    if "INCOME" in c.upper() and not c.upper().startswith("BP_") and not c.upper().startswith("CO_")
]
BP_INCOME_COLS_ALL = [c for c in sd_apr.columns if c.upper().startswith("BP_") and "INCOME" in c.upper()]
CO_INCOME_COLS_ALL = [c for c in sd_apr.columns if c.upper().startswith("CO_") and "INCOME" in c.upper()]

sd_apr["ent_units_row"] = sd_apr[ENT_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["bp_units_row"] = sd_apr[BP_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["co_units_row"] = sd_apr[CO_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)

above_mod_ent_all = [c for c in ENT_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_bp_all = [c for c in BP_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_co_all = [c for c in CO_INCOME_COLS_ALL if "ABOVE" in c.upper()]

sd_apr["ent_affordable_row"] = sd_apr["ent_units_row"] - sd_apr[above_mod_ent_all].sum(axis=1, numeric_only=True)
sd_apr["bp_affordable_row"] = sd_apr["bp_units_row"] - sd_apr[above_mod_bp_all].sum(axis=1, numeric_only=True)
sd_apr["co_affordable_row"] = sd_apr["co_units_row"] - sd_apr[above_mod_co_all].sum(axis=1, numeric_only=True)

production_history = (
    sd_apr[sd_apr[YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", YEAR_COL], as_index=False)
    .agg(
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
    )
    .rename(columns={YEAR_COL: "year"})
)
production_history["ent_affordable_share"] = production_history["ent_affordable_total"] / production_history["ent_units_total"]
production_history["bp_affordable_share"] = production_history["bp_affordable_total"] / production_history["bp_units_total"]
production_history["co_affordable_share"] = production_history["co_affordable_total"] / production_history["co_units_total"]

print(production_history.shape)
production_history.head()


In [ ]:
years_present = sorted(production_history["year"].dropna().unique())
years_missing = sorted(set(APR_YEARS) - set(years_present))
print("Years present:", years_present)
if years_missing:
    print("Years missing entirely from the pull:", years_missing)

coverage = (
    production_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage["missing_jurisdictions"] = coverage["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
coverage


In [ ]:
history_output_path = PROCESSED_DIR / f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv"
production_history.to_csv(history_output_path, index=False)
print("Saved:", history_output_path)


## APR Table A (applications)

Table A2 covers entitlements, permits, and completions -- but not
applications, which live in a separate table (Table A). Per HCD's own APR
instructions: *"an application is a formal submittal of a project for
approval... either an application for a discretionary entitlement, or...
the application for a building permit."*

Loaded the same way as Table A2 (CKAN action API, resolved by exact
resource name rather than a hardcoded ID).

In [ ]:
table_a_package_json = get_json(CKAN_PACKAGE_URL)  # same package as Table A2

# Exact-name match, not substring -- "table a" is a substring of "table a2",
# so the generic find_resource_download_url() helper would risk matching
# the wrong table here.
table_a_matches = [
    r for r in table_a_package_json["result"]["resources"]
    if r.get("name", "").strip().lower() == "apr table a"
    and r.get("format", "").upper() == "CSV"
]
if not table_a_matches:
    raise ValueError("No exact 'APR Table A' CSV resource found in the package.")
table_a_url = table_a_matches[0]["url"]
table_a_name = table_a_matches[0]["name"]
print("Using resource:", table_a_name)
print("Download URL:", table_a_url)

table_a_raw_path = RAW_DIR / "apr_table_a_raw.csv"
if not table_a_raw_path.exists():
    resp = requests.get(table_a_url, timeout=300)
    resp.raise_for_status()
    table_a_raw_path.write_bytes(resp.content)

apr_table_a_raw = pd.read_csv(table_a_raw_path, low_memory=False)
print(apr_table_a_raw.shape)
apr_table_a_raw.head()


In [ ]:
print(apr_table_a_raw.columns.tolist())

In [ ]:
TABLE_A_JUR_COL = "JURIS_NAME"
TABLE_A_YEAR_COL = "YEAR"

apr_table_a_raw["jur_clean"] = apr_table_a_raw[TABLE_A_JUR_COL].map(normalize_jurisdiction)
apr_table_a_raw[TABLE_A_YEAR_COL] = pd.to_numeric(apr_table_a_raw[TABLE_A_YEAR_COL], errors="coerce")

sd_apr_table_a = apr_table_a_raw[apr_table_a_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_table_a_target_year = sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr_table_a))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_table_a_target_year))
missing = sd_jur_keys - set(sd_apr_table_a_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


In [ ]:
# Confirmed: Table A's income-tier columns follow the same unprefixed
# pattern as Table A2's entitlement section (11 columns, DR/NDR split per
# tier except Above Moderate). Counts ALL applications submitted regardless
# of APPLICATION_STATUS (Pending/Approved/etc.) -- "applications submitted"
# is the metric, not "applications approved".
APPLICATION_INCOME_COLS = [
    "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
    "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
    "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    "LOW_INCOME_DR", "LOW_INCOME_NDR",
    "MOD_INCOME_DR", "MOD_INCOME_NDR",
    "ABOVE_MOD_INCOME",
]
above_mod_application = [c for c in APPLICATION_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_table_a_target_year["application_units_row"] = (
    sd_apr_table_a_target_year[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)
sd_apr_table_a_target_year["application_affordable_row"] = (
    sd_apr_table_a_target_year["application_units_row"]
    - sd_apr_table_a_target_year[above_mod_application].sum(axis=1, numeric_only=True)
)

applications_by_jurisdiction = (
    sd_apr_table_a_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
    )
)
applications_by_jurisdiction["application_affordable_share"] = (
    applications_by_jurisdiction["application_affordable_total"] / applications_by_jurisdiction["application_units_total"]
)

print(applications_by_jurisdiction.shape)
applications_by_jurisdiction


In [ ]:
# Sanity check: the income-tier sum should roughly match Table A's own
# TOT_PROPOSED_UNITS field, an independent count reported by jurisdictions.
# They won't be identical (TOT_PROPOSED_UNITS may include disapproved
# proposals differently), but a large divergence would flag a problem.
proposed_check = sd_apr_table_a_target_year.groupby("jur_clean", as_index=False).agg(
    tot_proposed_units=("TOT_PROPOSED_UNITS", "sum")
)
check = applications_by_jurisdiction.merge(proposed_check, on="jur_clean")
check["diff"] = check["application_units_total"] - check["tot_proposed_units"]
check[["jur_clean", "application_units_total", "tot_proposed_units", "diff"]]


## RHNA 6th Cycle targets

**RHNA progress basis:** the "units reported" figures in this dataset
(`rhna_reported_<tier>`, `rhna_pct_achieved_<tier>`, `rhna_remaining_<tier>`)
are based on **building permits issued**, not completed units / certificates
of occupancy. This is HCD's own methodology, not a choice made in this
notebook -- per HCD's APR guidance, only building-permit issuance counts
toward RHNA credit; entitlements and completions do not, even though both
are also tracked in the APR. This is a different measure than
`co_units_total` (completions) used elsewhere in this notebook for housing
production -- the two should never be conflated.

In [ ]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


In [ ]:
print(rhna_raw.columns.tolist())

In [ ]:
RHNA_JUR_COL = "Jurisdiction"  # confirm against columns above

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


## City of San Diego permits

In [ ]:
permits_dir = RAW_DIR.parent / "sandiego_permits"
permits_dir.mkdir(parents=True, exist_ok=True)

permits_frames = []
for label in ["active", "closed"]:
    raw_path = permits_dir / f"{label}_approvals_raw.csv"
    if not raw_path.exists():
        print(f"Missing: {raw_path}")
        print("Download from https://data.sandiego.gov/datasets/development-permits-set2/ and save it there.")
        continue
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

if permits_frames:
    sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
    print(sd_permits_raw.shape)
else:
    sd_permits_raw = pd.DataFrame()
sd_permits_raw.head()


In [ ]:
print(sd_permits_raw.columns.tolist())

In [ ]:
sd_permits_raw["APPROVAL_ISSUE_DATE"] = pd.to_datetime(sd_permits_raw["APPROVAL_ISSUE_DATE"], errors="coerce")

DU_TIER_COLS = [
    "APPROVAL_DU_EXTREMELY_LOW", "APPROVAL_DU_VERY_LOW", "APPROVAL_DU_LOW",
    "APPROVAL_DU_MODERATE", "APPROVAL_DU_ABOVE_MODERATE",
]
ADU_JADU_COLS = ["APPROVAL_ADU_TOTAL", "APPROVAL_JADU_TOTAL"]

# APPROVAL_DU_NET_CHANGE is unreliable (mostly null/zero) -- the actual
# per-project unit counts live in the income-tier DU columns, same pattern
# as APR's BP_*_INCOME fields.
sd_permits_raw["du_tier_total"] = sd_permits_raw[DU_TIER_COLS].fillna(0).sum(axis=1)
sd_permits_raw["adu_jadu_total"] = sd_permits_raw[ADU_JADU_COLS].fillna(0).sum(axis=1)

has_du_impact = (sd_permits_raw["du_tier_total"] != 0) | (sd_permits_raw["adu_jadu_total"] != 0)
in_target_year = sd_permits_raw["APPROVAL_ISSUE_DATE"].dt.year == TARGET_YEAR

sd_permits_housing = sd_permits_raw[has_du_impact & in_target_year].copy()
print(f"Housing-relevant permits, {TARGET_YEAR}:", len(sd_permits_housing))
print("Of", len(sd_permits_raw), "total raw rows")
sd_permits_housing[["PROJECT_TITLE", "JOB_BC_CODE_DESCRIPTION", "du_tier_total", "adu_jadu_total"]].head(10)


## CA DOF population & housing estimates (E-5)

In [ ]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

dof_raw["is_county_header"] = dof_raw["Total"].isna() & dof_raw["name"].str.contains("County", na=False)
dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["Total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: normalize_jurisdiction(COUNTY_JURISDICTION_NAME) if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
)
sd_dof["year"] = TARGET_YEAR

print("SD rows:", len(sd_dof))
missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_dof[["name", "Total", "Household", "Group Quarters"]]


## Census ACS (2020-2024 5-year estimates)

In [ ]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


## Check APR vs. City of SD permit overlap

In [ ]:
apr_sd_only = sd_apr_target_year[sd_apr_target_year["jur_clean"] == "san diego"]
apr_sd_units = apr_sd_only["bp_units_row"].sum()

city_permits_total = sd_permits_housing["du_tier_total"].sum() + sd_permits_housing["adu_jadu_total"].sum()

print(f"APR, City of San Diego, {TARGET_YEAR} permitted units: {apr_sd_units:.0f}")
print(f"City of SD permit system, {TARGET_YEAR} units (DU tiers + ADU/JADU): {city_permits_total:.0f}")
print(f"Ratio (city system / APR): {city_permits_total / apr_sd_units:.2f}" if apr_sd_units else "APR total is zero")


In [ ]:
# APN-based match check: how many City permit records share an APN with an
# APR-reported project for San Diego in the same year?
apr_sd_apns = set(apr_sd_only["APN"].dropna().astype(str).str.strip())
city_apns = set(sd_permits_housing["GIS_APN"].dropna().astype(str).str.strip())

overlap_apns = apr_sd_apns & city_apns
print(f"APR SD APNs ({TARGET_YEAR}): {len(apr_sd_apns)}")
print(f"City permit APNs ({TARGET_YEAR}): {len(city_apns)}")
print(f"APNs appearing in both: {len(overlap_apns)}")
print(f"Share of City permit APNs also in APR: {len(overlap_apns) / len(city_apns):.1%}" if city_apns else "no city APNs")


**Finding:** APR (City of San Diego rows) and the City of SD permits
dataset both stem from the same underlying City permitting activity --
the aggregate totals above are close in magnitude and a meaningful share
of APNs match directly. They should not be summed together when
calculating San Diego production totals; treat the City permits dataset
as a supplementary, more granular view of the same units APR already
counts for San Diego, not an additional source of units. This overlap
does not apply to the other 17 jurisdictions, where APR is the only source.


## Summary

In [ ]:
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "City of SD permits": sd_permits_housing if "sd_permits_housing" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


## Housing production categories & housing stock fields

In [ ]:
production_categories = pd.DataFrame([
    {"category": "Application submitted", "income_tier": "Acutely Low through Above Moderate (6 tiers)", "source": "APR Table A", "field": "ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOME (same 11-column pattern as Table A2's entitlement section)"},
    {"category": "Entitlement", "income_tier": "all tiers", "source": "APR Table A2", "field": "ENT_APPROVE_DT1, NO_ENTITLEMENTS"},
    {"category": "Building permit", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "BP_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "BP_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Very Low", "source": "APR Table A2", "field": "BP_VLOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Low", "source": "APR Table A2", "field": "BP_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Moderate", "source": "APR Table A2", "field": "BP_MOD_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "BP_ABOVE_MOD_INCOME"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "CO_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "CO_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Very Low", "source": "APR Table A2", "field": "CO_VLOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Low", "source": "APR Table A2", "field": "CO_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Moderate", "source": "APR Table A2", "field": "CO_MOD_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "CO_ABOVE_MOD_INCOME"},
    {"category": "RHNA target", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD"},
    {"category": "RHNA progress", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS"},
    {"category": "City permit approval", "income_tier": "Extremely Low / Very Low / Low / Moderate / Above Moderate", "source": "City of SD permits", "field": "APPROVAL_DU_EXTREMELY_LOW ... APPROVAL_DU_ABOVE_MODERATE"},
    {"category": "ADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_ADU_TOTAL (plus per-tier APPROVAL_ADU_* columns)"},
    {"category": "JADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_JADU_TOTAL (plus per-tier APPROVAL_JADU_* columns)"},
    {"category": "Preservation (existing affordable units retained)", "income_tier": "n/a", "source": "APR Table F (not yet loaded in this notebook)", "field": "see sd_apr_f_preservation_city_year.csv in dashboard prototype"},
])
production_categories.to_csv(DOCS_DIR / "housing_production_categories.csv", index=False)
production_categories


In [ ]:
stock_fields = pd.DataFrame([
    {"field": "Total population", "source": "DOF E-5", "column": "Total (population block)"},
    {"field": "Household population", "source": "DOF E-5", "column": "Household"},
    {"field": "Group quarters population", "source": "DOF E-5", "column": "Group Quarters"},
    {"field": "Total housing units", "source": "DOF E-5", "column": "Total (housing block)"},
    {"field": "Single detached units", "source": "DOF E-5", "column": "Single Detached"},
    {"field": "Single attached units", "source": "DOF E-5", "column": "Single Attached"},
    {"field": "2-4 unit buildings", "source": "DOF E-5", "column": "Two to Four"},
    {"field": "5+ unit buildings", "source": "DOF E-5", "column": "Five Plus"},
    {"field": "Mobile homes", "source": "DOF E-5", "column": "Mobile Homes"},
    {"field": "Occupied units", "source": "DOF E-5", "column": "Occupied"},
    {"field": "Vacancy rate", "source": "DOF E-5", "column": "Vacancy Rate"},
    {"field": "Persons per household", "source": "DOF E-5", "column": "Persons per Household"},
    {"field": "Total population", "source": "ACS 5-year", "column": "B01003_001E (population_total)"},
    {"field": "Total housing units", "source": "ACS 5-year", "column": "B25001_001E (housing_units_total)"},
    {"field": "Owner-occupied units", "source": "ACS 5-year", "column": "B25003_002E (owner_occupied)"},
    {"field": "Renter-occupied units", "source": "ACS 5-year", "column": "B25003_003E (renter_occupied)"},
])
stock_fields.to_csv(DOCS_DIR / "housing_stock_fields.csv", index=False)
stock_fields


**Notes on coverage:**

- DOF and ACS both report total population/housing units, but DOF breaks
  housing units out by structure type (single/multi/mobile) while ACS
  breaks out tenure (owner/renter) instead -- they're complementary, not
  duplicates, unlike the APR/City-permits overlap found above.
- The APR field names use `VLOW` for "Very Low" while RHNA6 uses `VLI` --
  same tier, different abbreviation between the two datasets.
- DR/NDR suffixes on APR income columns = Deed Restricted / Non-Deed
  Restricted (whether the affordability requirement is legally recorded
  on the property).
- Preservation (Table F) is not yet loaded into this notebook -- it exists
  as a processed file in the dashboard prototype repo but isn't part of
  this workstream's live pulls yet.


## Jurisdiction-year RHNA and housing production dataset

In [ ]:
# RHNA6 has no year column (cumulative cycle-to-date, not annual), so its
# target/progress numbers get joined onto each jurisdiction as static
# context rather than matched by year.
#
# Kept broken out by income category (not just totals) per the dashboard
# reorg guidance: "Show each jurisdiction's total RHNA allocation and its
# allocation by income category."
RHNA_TIERS = {
    "very_low": ("RHNA VLI", "VLI UNITS"),
    "low": ("RHNA LI", "LI UNITS"),
    "moderate": ("RHNA MOD", "MOD UNITS"),
    "above_moderate": ("RHNA ABOVE MOD", "ABOVE MOD UNITS"),
}

rhna_summary = sd_rhna6.copy()
tier_cols = ["jur_clean"]

for tier, (target_col, reported_col) in RHNA_TIERS.items():
    rhna_summary[f"rhna_target_{tier}"] = rhna_summary[target_col]
    rhna_summary[f"rhna_reported_{tier}"] = rhna_summary[reported_col]
    # Clipped at 0 -- overachieving one tier doesn't offset a shortfall in
    # another; each income tier is a separate obligation, not a shared pool.
    rhna_summary[f"rhna_remaining_{tier}"] = (
        rhna_summary[f"rhna_target_{tier}"] - rhna_summary[f"rhna_reported_{tier}"]
    ).clip(lower=0)
    rhna_summary[f"rhna_pct_achieved_{tier}"] = rhna_summary[f"rhna_reported_{tier}"] / rhna_summary[f"rhna_target_{tier}"]
    tier_cols += [
        f"rhna_target_{tier}", f"rhna_reported_{tier}",
        f"rhna_remaining_{tier}", f"rhna_pct_achieved_{tier}",
    ]

rhna_summary["rhna_target_total"] = rhna_summary[["RHNA VLI", "RHNA LI", "RHNA MOD", "RHNA ABOVE MOD"]].sum(axis=1)
rhna_summary["rhna_units_reported_total"] = rhna_summary[["VLI UNITS", "LI UNITS", "MOD UNITS", "ABOVE MOD UNITS"]].sum(axis=1)
# Sum of the already-clipped per-tier remainders, not (target_total -
# reported_total) clipped -- the same "no offsetting across tiers" logic
# applies to the total.
rhna_summary["rhna_remaining_total"] = rhna_summary[[f"rhna_remaining_{t}" for t in RHNA_TIERS]].sum(axis=1)
rhna_summary["rhna_pct_achieved"] = rhna_summary["rhna_units_reported_total"] / rhna_summary["rhna_target_total"]
tier_cols += ["rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total", "rhna_pct_achieved"]

rhna_summary = rhna_summary[tier_cols]

dof_summary = sd_dof.rename(columns={"Total": "population_total", "Household": "population_household"})[
    ["jur_clean", "population_total", "population_household"]
]

jurisdiction_year_dataset = (
    production_by_year
    .merge(applications_by_jurisdiction, on="jur_clean", how="left")
    .merge(rhna_summary, on="jur_clean", how="left")
    .merge(dof_summary, on="jur_clean", how="left")
)

print(jurisdiction_year_dataset.shape)
jurisdiction_year_dataset


In [ ]:
missing = sd_jur_keys - set(jurisdiction_year_dataset["jur_clean"].unique())
if missing:
    print("Jurisdictions missing from the combined table:", sorted(missing))
else:
    print("All 18 cities + County present.")

null_counts = jurisdiction_year_dataset.isna().sum()
null_counts[null_counts > 0]


## San Diego regional total

Per the dashboard reorg guidance: keep countywide totals clearly separate
from jurisdiction-level results rather than blending them into the same
table. This is built as its own object (not appended as a 20th row into
`jurisdiction_year_dataset`), summing all 18 cities plus the unincorporated
county area.

In [ ]:
assert len(jurisdiction_year_dataset) == 19, (
    f"Expected 18 cities + unincorporated county = 19 rows, got {len(jurisdiction_year_dataset)}"
)

RHNA_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]

SUM_COLS = [
    "application_units_total", "ent_units_total", "bp_units_total", "co_units_total",
    "application_affordable_total", "ent_affordable_total", "bp_affordable_total", "co_affordable_total",
    "project_rows", "rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total",
    "population_total", "population_household",
]
SUM_COLS += [f"rhna_target_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_reported_{t}" for t in RHNA_TIER_NAMES]
# rhna_remaining_<tier> is already clipped at 0 per jurisdiction, so summing
# it directly is correct -- one city's surplus in a tier can't offset
# another city's shortfall in that same tier.
SUM_COLS += [f"rhna_remaining_{t}" for t in RHNA_TIER_NAMES]

sd_region_total = jurisdiction_year_dataset[SUM_COLS].sum().to_frame().T
sd_region_total.insert(0, "jurisdiction", "San Diego Region (18 cities + Unincorporated County)")
sd_region_total.insert(1, "year", TARGET_YEAR)
sd_region_total.insert(2, "jurisdictions_included", len(jurisdiction_year_dataset))

# Share/percentage fields must be recalculated at the regional level, not
# summed or averaged directly -- summing per-city percentages would produce
# a meaningless number. Recalculated per tier as well as overall.
sd_region_total["application_affordable_share"] = sd_region_total["application_affordable_total"] / sd_region_total["application_units_total"]
sd_region_total["ent_affordable_share"] = sd_region_total["ent_affordable_total"] / sd_region_total["ent_units_total"]
sd_region_total["bp_affordable_share"] = sd_region_total["bp_affordable_total"] / sd_region_total["bp_units_total"]
sd_region_total["co_affordable_share"] = sd_region_total["co_affordable_total"] / sd_region_total["co_units_total"]
sd_region_total["rhna_pct_achieved"] = sd_region_total["rhna_units_reported_total"] / sd_region_total["rhna_target_total"]

for tier in RHNA_TIER_NAMES:
    sd_region_total[f"rhna_pct_achieved_{tier}"] = (
        sd_region_total[f"rhna_reported_{tier}"] / sd_region_total[f"rhna_target_{tier}"]
    )

sd_region_total


In [ ]:
# Sanity check: regional total should exactly equal the sum of the 19
# individual jurisdiction rows, not an independently-sourced number.
check = jurisdiction_year_dataset["bp_units_total"].sum() == sd_region_total["bp_units_total"].iloc[0]
print("Regional bp_units_total matches sum of jurisdiction rows:", check)

region_output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv"
sd_region_total.to_csv(region_output_path, index=False)
print("Saved:", region_output_path)


## Metric dictionary

In [ ]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "application_units_total",
        "source_table": "HCD APR Table A",
        "source_variables": "Sum of unprefixed *_INCOME_* columns, all APPLICATION_STATUS values included",
        "definition": "Total housing units in applications submitted, all income tiers, per jurisdiction-year. Counts all applications regardless of approval status -- this is 'submitted', not 'approved'. Cross-checked against Table A's own TOT_PROPOSED_UNITS field",
    },
    {
        "output_metric": "ent_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of unprefixed *_INCOME_* columns (not BP_ or CO_ prefixed)",
        "definition": "Total housing units with a planning entitlement approved, all income tiers, per jurisdiction-year. Kept fully separate from bp_units_total and co_units_total -- never summed together",
    },
    {
        "output_metric": "bp_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns",
        "definition": "Total housing units with a building permit issued, all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "co_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns",
        "definition": "Total housing units with a certificate of occupancy (completed), all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "rhna_target_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD",
        "definition": "Assigned RHNA target units, kept separate per income tier (very_low, low, moderate, above_moderate) for the 6th Cycle planning period, per jurisdiction -- plus rhna_target_total for the summed value",
    },
    {
        "output_metric": "rhna_reported_<tier> / rhna_pct_achieved_<tier> / rhna_remaining_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS",
        "definition": "BASED ON BUILDING PERMITS ISSUED, not completed units -- this is HCD's own RHNA-credit methodology, not a choice made here. Kept separate per income tier, per jurisdiction. Cumulative cycle-to-date, not year-by-year. Do not conflate with co_units_total (completions), which is a different measure used for housing production, not RHNA progress. rhna_units_reported_total / rhna_pct_achieved / rhna_remaining_total are the summed/overall equivalents",
    },
    {
        "output_metric": "sd_permit_du_by_tier",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERATE/ABOVE_MODERATE",
        "definition": "Dwelling units per approval, broken out by income tier, City of San Diego only. Separate ADU (APPROVAL_ADU_*) and JADU (APPROVAL_JADU_*) columns exist alongside standard units",
    },
    {
        "output_metric": "sd_permit_stage",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, approval_status",
        "definition": "This dataset tracks permit issuance and closure only -- it has no certificate-of-occupancy / completion field equivalent to APR's CO_* columns",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


## Export

In [ ]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
jurisdiction_year_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)


## Validation against dashboard prototype

In [ ]:
def find_sibling_repo(repo_name: str, search_depth: int = 3) -> Path | None:
    """
    Look for a sibling clone of another repo near this workstream, without
    assuming a fixed folder depth -- works regardless of what each person
    named their local GitHub folder, as long as both repos live under the
    same parent directory somewhere.
    """
    candidates = [ROOT, *ROOT.parents][:search_depth + 2]
    for base in candidates:
        match = base / repo_name
        if match.exists():
            return match
        # also check one level of siblings, in case repos sit in a shared
        # "GitHub" folder rather than directly next to each other
        if base.parent.exists():
            for sibling in base.parent.iterdir():
                if sibling.name == repo_name and sibling.is_dir():
                    return sibling
    return None


baseline_repo = find_sibling_repo("housing-dashboard-prototype")

if baseline_repo is None:
    print(
        "Could not find a local clone of housing-dashboard-prototype near this repo. "
        "Clone it (git clone https://github.com/laurenthanhvo/housing-dashboard-prototype.git) "
        "into the same parent folder as this repo, then re-run this cell."
    )
else:
    BASELINE_PATH = baseline_repo / "data" / "processed" / "sd_apr_a2_city_year_supply.csv"
    print("Found baseline repo at:", baseline_repo)


In [ ]:
if baseline_repo is not None and BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    baseline_years = sorted(baseline["year"].unique())
    if TARGET_YEAR not in baseline_years:
        print(
            f"Baseline (dashboard prototype) only has data through {max(baseline_years)}; "
            f"it does not yet include {TARGET_YEAR}. Nothing to validate against yet -- "
            "this notebook's pull is ahead of the dashboard prototype's data."
        )
    else:
        comparison = production_by_year.merge(
            baseline, on=["jur_clean", "year"], suffixes=("_fresh", "_baseline"), how="inner",
        )
        failed_checks = comparison[
            comparison["bp_units_total_fresh"] != comparison["bp_units_total_baseline"]
        ]
        print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches.")
        display(failed_checks[["jur_clean", "bp_units_total_fresh", "bp_units_total_baseline"]])
        comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
elif baseline_repo is not None:
    print(f"Repo found but expected file is missing: {BASELINE_PATH}")


## Missing data, unclear fields, and source limitations

In [ ]:
data_quality_log = pd.DataFrame([
    # -- Missing data --
    {"type": "Missing data", "source": "APR Table A2", "item": "bp_affordable_share / co_affordable_share",
     "note": "NaN when bp_units_total or co_units_total is 0 for that jurisdiction-year (0/0), not a data error -- means zero permits/completions that year, not unknown affordability"},
    {"type": "Missing data", "source": "APR Table A2", "item": "NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST_NAME, PRIOR_APN",
     "note": "Conditional fields -- only populated for specific project types (e.g. DR_TYPE only for deed-restricted units). Sparse by design, not incomplete"},
    {"type": "Missing data", "source": "City of SD permits", "item": "ADU/JADU and income-tier DU columns",
     "note": "Blank on non-residential permit rows (electrical, plumbing, signage, etc.) -- expected, filtered out of sd_permits_housing"},
    {"type": "Missing data", "source": "APR Table F (preservation)", "item": "entire table",
     "note": "Not yet loaded into this workstream. Exists as a processed file in the dashboard prototype repo (sd_apr_f_preservation_city_year.csv) but not pulled live here"},
    {"type": "Missing data", "source": "This workstream", "item": "NOAH (naturally occurring affordable housing) loss estimate",
     "note": "No published dataset exists for this -- would need to be derived from ACS rent + building-age data. Not started"},
    {"type": "Missing data", "source": "HCD APR (general)", "item": "current reporting year, some jurisdictions",
     "note": "Jurisdictions can file late; a given year's data may be incomplete for months after the April 1 deadline. Check the 'not found' warning printed when loading before trusting a fresh pull"},
    {"type": "Missing data", "source": "HCD APR", "item": "units under construction",
     "note": "Not tracked anywhere in HCD's APR data -- no table captures this milestone. Application, entitlement, permit, and completion stages are all available (Table A and Table A2), but under-construction status is not"},
    {"type": "Missing data", "source": "APR Table A2, City of San Diego", "item": "ent_units_total near-zero relative to bp_units_total",
     "note": "San Diego reported only 3 entitled units in 2025 against 7,842 permitted units -- an unusually large gap for the county's largest jurisdiction (other cities show entitlement counts in the hundreds to low thousands, more in line with their permit volume). Likely a self-reporting gap in San Diego's own APR submission for the entitlement stage specifically, not evidence that entitlement activity actually stopped. Worth flagging to the jurisdiction/HCD rather than treating as a real zero"},

    # -- Unclear fields --
    {"type": "Unclear field", "source": "APR Table A2", "item": "*_DR / *_NDR column suffixes",
     "note": "Deed Restricted / Non-Deed Restricted (whether the affordability requirement is legally recorded on the property). Not explained in the CSV itself -- confirmed from HCD's separate Table A2 data dictionary (.docx)"},
    {"type": "Unclear field", "source": "APR Table A2 vs. RHNA6", "item": "VLOW vs. VLI",
     "note": "Same income tier (Very Low Income), different abbreviation between two HCD datasets. Easy to miss if joining/comparing by tier name"},
    {"type": "Unclear field", "source": "RHNA6 progress report", "item": "what \"units reported\" actually measures",
     "note": "RHNA progress in this dataset is based on BUILDING PERMITS ISSUED, not completed units -- confirmed via HCD's own APR guidance (\"only building permits are used for the purposes of determining progress towards RHNA\"). Entitlements and completions are tracked elsewhere in the APR but do not count toward RHNA credit. Not stated in the RHNA6 file's own column headers, easy to assume otherwise"},
    {"type": "Unclear field", "source": "City of SD permits", "item": "APPROVAL_DU_NET_CHANGE",
     "note": "Misleadingly named -- sums to exactly 0 across a full year of data, unreliable. Real unit counts live in the separate income-tier APPROVAL_DU_* columns instead"},
    {"type": "Resolved / corrected", "source": "APR Table A2", "item": "NO_ENTITLEMENTS / NO_BUILDING_PERMITS / NO_OTHER_FORMS_OF_READINESS",
     "note": "CORRECTED: previously mischaracterized as flags in an earlier version of this log. Per HCD's official APR instructions, these mean \"Number Of\" -- auto-populated total-unit-count fields calculated by HCD from the same per-tier income columns this notebook sums independently. Cross-check against ent_units_total / bp_units_total / co_units_total matches exactly for 18 of 19 jurisdictions (bp and co match all 19/19)"},
    {"type": "Missing data", "source": "APR Table A2", "item": "Escondido ent_units_total vs. NO_ENTITLEMENTS mismatch (55-unit gap)",
     "note": "Our sum (421) does not match HCD's own auto-populated NO_ENTITLEMENTS total (366) for Escondido specifically -- every other jurisdiction using identical logic matches exactly, so this points to an inconsistency in Escondido's raw APR submission itself (e.g. a late correction to per-tier figures that did not refresh HCD's auto-populated field), not a calculation error in this notebook"},
    {"type": "Missing data", "source": "APR Table A", "item": "application_units_total vs. TOT_PROPOSED_UNITS small discrepancies",
     "note": "14 of 19 jurisdictions match exactly; 5 show small gaps (largest: Escondido, 88 units / ~16% relative). Likely reflects units reported in TOT_PROPOSED_UNITS without a corresponding income-tier breakdown yet, rather than a calculation error -- see the cross-check cell in the Table A section"},
    {"type": "Unclear field", "source": "HCD / DOF / dashboard prototype", "item": "County jurisdiction naming",
     "note": "Appears as 'Unincorporated' (DOF), 'SAN DIEGO COUNTY' (APR/RHNA), and 'County of San Diego' in various places -- required manual normalization to a single consistent key ('san diego county') across this notebook"},

    # -- Source limitations --
    {"type": "Source limitation", "source": "APR / RHNA (HCD)", "item": "self-reported",
     "note": "Not independently verified by HCD. Quality, completeness, and filing timeliness vary by jurisdiction"},
    {"type": "Source limitation", "source": "RHNA6 progress report", "item": "cumulative only",
     "note": "No year-by-year breakdown -- reports cycle-to-date totals only (2021-2029), can't see year-over-year pace toward the target from this file alone"},
    {"type": "Source limitation", "source": "City of SD permits", "item": "single-jurisdiction coverage",
     "note": "Only covers the City of San Diego, not the other 17 jurisdictions. Also overlaps with APR for San Diego specifically (~1.04 ratio, 93% APN match) -- don't sum the two for San Diego totals"},
    {"type": "Source limitation", "source": "DOF E-5", "item": "annual point-in-time estimate",
     "note": "January 1 snapshot, not real-time. Uses different methodology than ACS, so the two won't match exactly even for the same year"},
    {"type": "Source limitation", "source": "Census ACS", "item": "one year behind target year",
     "note": "Newest available vintage is 2020-2024 (\"2024\" data) while APR/DOF/permits target 2025 -- ACS structurally cannot produce 2025 data until ~Dec 2026/Jan 2027. Also a 5-year rolling estimate with margins of error, largest for small jurisdictions like Del Mar (~3,900 population)"},
    {"type": "Source limitation", "source": "Dashboard prototype (housing-dashboard-prototype repo)", "item": "stale baseline for validation",
     "note": "The prototype's sd_apr_a2_city_year_supply.csv only covers 2018-2024 -- it has no 2025 data yet, so this notebook's 2025 pull currently has nothing to validate against. Not an error in this notebook; the prototype simply hasn't been refreshed with 2025 APR data"},
])

data_quality_log.to_csv(DOCS_DIR / "data_quality_limitations.csv", index=False)
data_quality_log
